# 12 · Skill router: load only the sub-agent that fits

## The problem

Agent frameworks give an agent a **roster** of skills or sub-agents: one reads PDFs, one writes SQL, one builds slides,
and so on. Stuffing *all* of their instructions into the context on every turn is slow, expensive, and confuses the
model. You want to load **only the skill that fits the task**.

That's easy for "translate this into Japanese". It gets hard in three places:
- **look-alikes**: "is the SQL in this pull request right?" could be the *code reviewer* or the *SQL analyst*;
- **multi-skill tasks**: "turn this PDF into a slide deck" needs the PDF reader **and** the slide builder;
- **nothing fits**: "book me a table for two" isn't any of them.

## Why Jev

Routing happens on **every** task, before any real work starts, so it has to be fast and cheap. Jev is a **System One**
model: it doesn't write text, it returns **typed judgments with calibrated probabilities** in about 100 ms, for a
fraction of a cent. Two of those judgments are all a router needs:
- a `Choice` over the skill descriptions, returning a **probability for every skill**, not just a winner,
- a `Noul` (yes/no probability): *does this task need more than one skill?*

Because the probabilities are calibrated, code can read the **gap between the top two skills** to spot ambiguity.

## What we'll build

```
task ──► Jev: ONE call, two questions ──► plain-Python policy ──────────► action
          • skill        (Choice over roster)   • top skill is "none"  → NO SKILL
          • needs_several (Noul)                 • needs_several high   → MULTI-STEP (top 2)
                                                 • top-1 minus top-2 small → AMBIGUOUS
                                                 • otherwise            → LOAD ONE
```

## What you'll learn
- How a `Choice` returns a full probability distribution, and why the **runner-up** is as useful as the winner.
- Passing a **dict as `state`**, and referring to its keys with backticks in instructions.
- Adding a `"none"` option so the router can say *nothing fits* instead of forcing a bad match.

*From the use-case catalog: [18 practical Jev use cases for AI agents](https://huggingface.co/blog/karmen-beatapi/18-practical-jev-use-cases-for-ai-agents), #4.*

> **How to run this notebook**
> 1. Pick the **`.venv`** kernel (top right in VS Code, *Kernel → Change kernel* in Jupyter).
> 2. Run the cells **top to bottom** (or *Run All*). Every cell builds on the ones above it.
> 3. If you see `NameError: name '...' is not defined`, the kernel was restarted (switching kernels does this) and forgot the earlier cells. Click **Run All** or *Run All Above*, then continue.

## Step 0 · Setup

This cell:
- loads the API key (`TYPESAFE_API_KEY`) from the `.env` file at the repo root,
- creates the Jev client, `jev`, that every later cell uses,
- imports the Jev question types (`Choice` and `Noul` are the two we use here),
- defines `bar()`, a tiny helper that draws a 0..1 probability as a text bar so you can *see* the numbers.

In [1]:
import os, re
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient, Noul, NoulCriteria, Score, Choice

# Look for a .env file in this folder or any parent folder
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break

jev = TypeSafeClient()   # the judge: fast, cheap, typed answers


def bar(p, width=20):
    """A unicode progress bar for a 0..1 value."""
    filled = round(max(0.0, min(1.0, p)) * width)
    return "█" * filled + "·" * (width - filled)


print("Jev client ready")

Jev client ready


## Step 1 · The roster and the two questions

`SKILLS` is the roster: a short, plain-English description per skill. Those descriptions **are** the router's whole
world, so write them the way you'd brief a new colleague. Note the explicit `"none"` option: without it, a `Choice` has
to pick *something*, even for a task no skill can do.

We define two questions:

| Key | Type | What it returns | Why the router needs it |
|---|---|---|---|
| `skill` | `Choice` | `.choice` (the winner), `.confidence`, and `.probabilities`, a dict with one probability per skill | which skill fits best, and how close the runner-up is |
| `needs_several` | `Noul` | `.noul`, the probability that the answer is **yes** (0 to 1) | multi-skill tasks need a plan, not a single skill |

**`state` can be a string or a dict.** Here it will be a dict `{"task": ..., "skills": ...}`. Inside instructions,
a name in backticks such as `` `task` `` or `` `skills` `` refers to that key of the state.

This cell only **defines** things. Nothing is sent to Jev yet.

In [2]:
SKILLS = {
    "pdf_extractor":  "Read PDF files and pull out text, tables or fields",
    "sql_analyst":    "Write and run SQL against the company database to answer data questions",
    "web_researcher": "Search the public web and summarize sources",
    "slide_builder":  "Create presentation slides from content",
    "code_reviewer":  "Review a code diff for bugs and style problems",
    "translator":     "Translate text between languages",
    "none":           "None of these skills fits the task",
}

ROUTER = {
    "skill": Choice(instructions="Which single skill best fits `task`?", criteria=SKILLS),
    "needs_several": Noul(instructions="To finish `task`, would the agent have to use TWO OR MORE different skills "
                                       "from `skills`, one after another? Answer no if one skill alone can do it."),
}
print(f"{len(SKILLS)} skills in the roster, {len(ROUTER)} questions defined:", list(ROUTER))

7 skills in the roster, 2 questions defined: ['skill', 'needs_several']


## Step 2 · Ask once and read the raw answer

`jev.system_one(state=..., questions=...)` sends the task plus **both** questions in **one** request. Answers come back
grouped by type: `r.choices["skill"]` and `r.nouls["needs_several"]`.

We try a task that obviously needs two skills: read a PDF, then build slides from it.

In [3]:
r = jev.system_one(state={"task": "Turn the attached quarterly-report PDF into a 5-slide summary deck", "skills": SKILLS},
                   questions=ROUTER)

skill = r.choices["skill"]
print(f"winner: {skill.choice}   (confidence {skill.confidence:.2f})\n")
for name, p in sorted(skill.probabilities.items(), key=lambda kv: -kv[1]):
    print(f"  {name:<15} {bar(p)} {p:.2f}")
print(f"\nneeds_several  {bar(r.nouls['needs_several'].noul)} {r.nouls['needs_several'].noul:.2f}")

winner: slide_builder   (confidence 0.86)

  slide_builder   ██████████████████·· 0.88
  pdf_extractor   ██·················· 0.12
  sql_analyst     ···················· 0.00
  web_researcher  ···················· 0.00
  code_reviewer   ···················· 0.00
  translator      ···················· 0.00
  none            ···················· 0.00

needs_several  ██████████████████·· 0.92


**What to notice**
- `slide_builder` wins, but `pdf_extractor` is a clear **runner-up**. A plain "pick the winner" router would throw that
  information away, and the agent would try to build slides without ever reading the PDF.
- The skills that have nothing to do with the task (SQL, translation, code review) sit near zero, and so does `none`.
- `needs_several` is high: Jev recognises this is a two-step job.
- Both answers came from **one** request.

## Step 3 · The routing policy, in plain Python

Jev only produced **numbers**. Every *decision* happens here, in code you can read, test and change. The rules are
checked in order, and the first one that matches wins:

| Rule | Condition | Action |
|---|---|---|
| 1 | top skill is `none` | **NO SKILL**: the main agent handles it (or politely declines) |
| 2 | `needs_several > SEVERAL_AT` (0.60) | **MULTI-STEP**: load the top two skills and let the planner sequence them |
| 3 | top-1 minus top-2 `< MARGIN` (0.30) | **AMBIGUOUS**: ask the user, or try top-1 with top-2 as fallback |
| 4 | otherwise | **LOAD ONE**: load just the winning skill |

The **margin** rule is the key idea: a 0.55 vs 0.45 split and a 0.99 vs 0.01 split have the same winner, but they are
very different situations.

`route()` starts with a guard: if the kernel was restarted, `ROUTER` won't exist, and you get a clear message instead of
a confusing `NameError` halfway through.

> **Workshop note:** two small changes from the first draft. (1) `MARGIN` went from 0.20 to **0.30**, i.e. "the winner
> needs at least a 65/35 split before we commit". At 0.20 the look-alike SQL task below flipped between AMBIGUOUS and
> LOAD ONE from run to run, because its gap sits right around 0.20. A threshold should sit in a *gap* in the scores,
> not on top of a cluster. (2) The multi-step plan is listed **by probability**, not in execution order (slides can
> outrank the PDF reader even though the PDF must be read first), so the action now says *let the planner sequence
> them* instead of promising an order.

In [4]:
MARGIN = 0.30        # top-1 minus top-2 below this = ambiguous (winner needs at least a 65/35 split)
SEVERAL_AT = 0.60    # needs_several above this = multi-step task


def route(task, client):
    if "ROUTER" not in globals():
        raise RuntimeError("ROUTER is missing: the kernel restarted. Use 'Run All' (or run the cells above) first.")

    r = client.system_one(state={"task": task, "skills": SKILLS}, questions=ROUTER)
    ranked = sorted(r.choices["skill"].probabilities.items(), key=lambda kv: -kv[1])
    (top, p1), (second, p2) = ranked[0], ranked[1]
    several = r.nouls["needs_several"].noul

    if top == "none":
        plan, action = [], "NO SKILL: handle in the main agent"
    elif several > SEVERAL_AT:
        plan, action = [s for s, p in ranked[:2] if s != "none"], "MULTI-STEP: load both, planner sequences"
    elif p1 - p2 < MARGIN:
        plan, action = [top, second], "AMBIGUOUS: ask, or top-1 with fallback"
    else:
        plan, action = [top], "LOAD ONE"
    return {"ranked": ranked, "several": several, "margin": p1 - p2, "plan": plan, "action": action}


d = route("Translate our README into Japanese", jev)
print(d["action"], d["plan"], f"(margin {d['margin']:.2f}, needs_several {d['several']:.2f})")

LOAD ONE ['translator'] (margin 1.00, needs_several 0.15)


## Step 4 · Route a mixed batch

Seven tasks chosen to hit every branch: clear single-skill tasks, a two-skill task, a **look-alike** (SQL inside a
pull request: code reviewer or SQL analyst?), and a task that fits no skill at all. For each we print the action, the
plan, the gap between the top two skills, and `needs_several`.

In [5]:
TASKS = [
    "How many orders did we ship last month, by region?",
    "Check this pull request for an off-by-one bug",
    "Turn the attached quarterly-report PDF into a 5-slide summary deck",
    "Find recent news articles about solid-state batteries",
    "Translate our README into Japanese",
    "Is the SQL in this pull request going to return the right numbers?",
    "Book me a table for two at 8pm",
]
for t in TASKS:
    d = route(t, jev)
    print(f"{t[:58]:<58} -> {d['action'].split(':')[0]:<10} {d['plan']}")
    print(f"{'':<58}    margin {d['margin']:.2f}   needs_several {d['several']:.2f}")

How many orders did we ship last month, by region?         -> LOAD ONE   ['sql_analyst']
                                                              margin 1.00   needs_several 0.23


Check this pull request for an off-by-one bug              -> LOAD ONE   ['code_reviewer']
                                                              margin 1.00   needs_several 0.08


Turn the attached quarterly-report PDF into a 5-slide summ -> MULTI-STEP ['slide_builder', 'pdf_extractor']
                                                              margin 0.76   needs_several 0.93


Find recent news articles about solid-state batteries      -> LOAD ONE   ['web_researcher']
                                                              margin 1.00   needs_several 0.11


Translate our README into Japanese                         -> LOAD ONE   ['translator']
                                                              margin 1.00   needs_several 0.18


Is the SQL in this pull request going to return the right  -> AMBIGUOUS  ['sql_analyst', 'code_reviewer']
                                                              margin 0.10   needs_several 0.28


Book me a table for two at 8pm                             -> NO SKILL   []
                                                              margin 0.98   needs_several 0.16


**What to notice**
- The clear single-skill tasks (orders by region, off-by-one, news search, translation) are **LOAD ONE** with a huge
  margin: there's no doubt, so we load exactly one skill.
- The PDF-to-deck task is **MULTI-STEP**: `needs_several` is high, so both skills get loaded.
- The SQL-in-a-pull-request task is the look-alike. `code_reviewer` and `sql_analyst` split the probability, the margin
  is small, and the router says **AMBIGUOUS** instead of guessing. That's the runner-up earning its keep.
- "Book me a table" goes to **NO SKILL**, thanks to the explicit `none` option.

## Step 5 · Give it a face

Everything from here on is **presentation only**. This cell is a small style kit (colours, `box`, `badge`, `meter`) and
`form_app()`, which wires text boxes and a button to any function that returns HTML. `route_html()` turns one routing decision into a card with the top four skill probabilities.
**The decision logic above is untouched**: the UI only calls the same functions you've already seen run.

In [6]:
import ipywidgets as widgets
from IPython.display import HTML, display

INK, SLATE, CORAL, SAGE, AMBER, RED, LINE = "#1F1E1D", "#5E6B7A", "#D97757", "#1F7A3D", "#C98A00", "#C0392B", "#DEDACE"
CARD = f"background:#FAF9F5;border:1px solid {LINE};border-radius:10px;padding:14px 16px;margin:8px 0;"
BADGE = "display:inline-block;padding:2px 10px;border-radius:12px;color:white;font-weight:600;font-size:12px;"


def box(inner):
    return f'<div style="font-family:-apple-system,Segoe UI,sans-serif;color:{INK};max-width:860px;">{inner}</div>'


def badge(text, color):
    return f'<span style="{BADGE}background:{color};">{text}</span>'


def meter(label, p, width=160):
    """A labelled horizontal probability bar."""
    col = RED if p > 0.7 else AMBER if p > 0.4 else SAGE
    return (f'<div style="display:flex;align-items:center;gap:8px;font-size:12px;margin:2px 0;">'
            f'<span style="width:190px;color:{SLATE};">{label}</span>'
            f'<span style="width:{width}px;height:8px;background:#ECE9E0;border-radius:4px;overflow:hidden;">'
            f'<span style="display:block;height:8px;width:{p * 100:.0f}%;background:{col};"></span></span>'
            f'<b>{p:.2f}</b></div>')


def form_app(fields, render, button="Run"):
    """fields: list of (name, label, default, rows). render(**values) returns HTML."""
    boxes = {}
    for name, label, default, rows in fields:
        w = (widgets.Textarea(value=default, rows=rows, layout=widgets.Layout(width="90%"))
             if rows > 1 else widgets.Text(value=default, layout=widgets.Layout(width="90%")))
        boxes[name] = w
    btn = widgets.Button(description=button, button_style="warning")
    out = widgets.Output()

    def go(_):
        with out:
            out.clear_output()
            display(HTML(box("<i>thinking...</i>")))
            html = render(**{k: w.value.strip() for k, w in boxes.items()})
            out.clear_output(wait=True)
            display(HTML(html))

    btn.on_click(go)
    rows = [widgets.VBox([widgets.HTML(f"<b>{label}</b>"), boxes[name]]) for name, label, _, _ in fields]
    display(widgets.VBox(rows + [btn]), out)

The next cell defines `route_html()` and renders two cards from the batch above (the two-skill task and the news search)
so you can see what the app looks like before typing your own tasks.

In [7]:
def route_html(task):
    d = route(task, jev)
    col = SAGE if d["action"] == "LOAD ONE" else AMBER if d["plan"] else SLATE
    bars = "".join(meter(n, p) for n, p in d["ranked"][:4])
    plan = " + ".join(f"<code>{s}</code>" for s in d["plan"]) or "<i>no skill loaded</i>"
    return box(f'<div style="{CARD}border-left:5px solid {col};"><div style="margin-bottom:6px;">{task}</div>'
               f'{badge(d["action"].split(":")[0], col)} &nbsp;{plan}'
               f'<div style="margin-top:8px;">{bars}{meter("needs several skills", d["several"])}</div></div>')


display(HTML(box("".join(route_html(t) for t in TASKS[2:4]))))

## Step 6 · Try it yourself

Type a task and press **Route** (needs a live kernel). Good ones to try in a workshop:
- `Summarize this Spanish PDF contract in English` → **MULTI-STEP**: PDF reader + translator.
- `Make me a 3-slide deck about our Q3 sales by region from the database` → **MULTI-STEP**: SQL + slides.
- `Look over this SQL query and tell me if the JOIN is wrong` → another look-alike; watch the margin, it hovers near
  the line, so you may see **AMBIGUOUS** or **LOAD ONE** `code_reviewer`.
- `What's the weather in Paris?` → no skill fits, so expect **NO SKILL** (or a low-margin `web_researcher`).

In [8]:
form_app([("task", "Task", "Summarize this Spanish PDF contract in English", 2)], route_html, "Route")

Output()

## Recap and exercises

**Key ideas**
1. A router runs on **every** task, so it must be cheap and fast. One Jev call answers both questions in ~100 ms.
2. **Use the runner-up.** A `Choice` gives a probability per option; the gap between the top two is a direct measure of
   ambiguity.
3. **Give the model a way out.** An explicit `none` option stops the router forcing a bad match.
4. **The model measures, your code decides.** `MARGIN` and `SEVERAL_AT` are ordinary constants: put them in the gaps
   between your measured scores.

**Exercises**
- Set `MARGIN = 0.10` and re-run Step 4. Which task stops being AMBIGUOUS? Is that what you want?
- Add a `"calendar_agent": "Book meetings, tables and appointments"` skill and re-route *Book me a table for two*.
- Make the two descriptions more distinct (for example, `code_reviewer`: *"... a code diff, not data questions"*) and
  watch the look-alike's margin change. Wording is the model's whole world.
- Ask a third question, a `Choice` over the roster: *"Which skill must run FIRST?"*, and use it to order the MULTI-STEP
  plan.